# NILM-Engine — Run 6: Incremental + house_mask (early stopping 수정)

| 항목 | 값 |
|------|----|
| 실험 전략 | 1주차씩 누적 추가학습 (EXP1→2→3→4) + house_mask 조건화 |
| Early stopping 수정 | `val_mae` 단독 모니터링 (Run5 f1_cls 포화 버그 수정) |
| Best 선택 | 주차별 val MAE 추적 → 최저 MAE 주차 체크포인트 사용 |
| 가설 | house_mask로 미등록 가전 억제 + incremental로 데이터 효율 극대화 |
| Train | house_011,015,016,017,033,039,054,063 (주차 누적) |
| Val | house_049 (전체 기간) |
| Test | house_067 (전체 기간) |
| 비교 기준 | Run3 val=39.32W/test=44.14W · Run4 val=44.09W/test=46.12W · Run5 val=1256W(버그)/test=47.09W |
---

In [ ]:
!pip install -q gudhi

In [ ]:
from google.colab import auth
auth.authenticate_user()
print("GCP 인증 완료")

In [ ]:
import gcsfs
gcs = gcsfs.GCSFileSystem()
print("GCS 연결 완료")

In [ ]:
SPLIT = {
    "train": ["house_011", "house_015", "house_016", "house_017",
              "house_033", "house_039", "house_054", "house_063"],
    "val":   ["house_049"],
    "test":  ["house_067"],
}
BUCKET_PREFIX = "ax-nilm-data-dhwang0803-us/nilm/training_dev10"
LABEL_PATH    = "ax-nilm-data-dhwang0803-us/nilm/labels/training.parquet"
MODELS        = ["cnn_tda"]

print(f"Train {len(SPLIT['train'])}개 / Val {len(SPLIT['val'])}개 / Test {len(SPLIT['test'])}개")

In [ ]:
import os, sys

REPO_DIR = "/content/ax_nilm"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/dhwang0803-glitch/ax_nilm {REPO_DIR} -q
    print("클론 완료")
else:
    !git -C {REPO_DIR} pull -q && echo "최신화 완료"

SRC_DIR     = f"{REPO_DIR}/nilm-engine/src"
SCRIPTS_DIR = f"{REPO_DIR}/nilm-engine/scripts"
CFG_DIR     = f"{REPO_DIR}/nilm-engine/config"

for d in [SRC_DIR, SCRIPTS_DIR]:
    if d not in sys.path:
        sys.path.insert(0, d)
print("경로 설정 완료")

In [ ]:
import yaml

from acquisition.gcs_loader import (
    list_channels_gcs, get_house_start_date_gcs,
    load_channel_data_gcs, get_appliance_name_gcs,
    GCSNILMDataset, build_house_mask_dict,
)
from acquisition.preprocessor import PowerScaler
from train_model import (
    build_model, masked_weighted_mse, evaluate, train_one_epoch,
    compute_pos_weight, _NILMDatasetWithTDA, APPLIANCE_LABELS,
)

with open(f"{CFG_DIR}/train.yaml")   as f: TRAIN_CFG   = yaml.safe_load(f)
with open(f"{CFG_DIR}/dataset.yaml") as f: DATASET_CFG = yaml.safe_load(f)

print("EXP 설정:")
for exp in ["EXP1", "EXP2", "EXP3", "EXP4"]:
    cfg = TRAIN_CFG["experiments"][exp]
    print(f"  {exp}: week={cfg['week']}  resume_from={cfg['resume_from']}")
print("모든 모듈 import 완료")

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

CKPT_DIR    = Path("/content/drive/MyDrive/nilm_run6/checkpoints")
RESULTS_DIR = Path("/content/drive/MyDrive/nilm_run6/results")
CACHE_DIR   = Path("/content/drive/MyDrive/nilm/cache")  # Run4/5 캐시 재사용
CKPT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print(f"체크포인트 → {CKPT_DIR}")
print(f"결과       → {RESULTS_DIR}")
print(f"캐시       → {CACHE_DIR}")

## 가구별 등록 가전 마스크 빌드

In [ ]:
from classifier.label_map import APPLIANCE_LABELS as _LABELS

HOUSE_MASK = build_house_mask_dict(gcs, LABEL_PATH)

print(f"마스크 빌드 완료: {len(HOUSE_MASK)}가구")
print("\n=== train/val/test 가구별 마스크 (등록 가전 수) ===")
all_houses = SPLIT["train"] + SPLIT["val"] + SPLIT["test"]
for h in all_houses:
    mask = HOUSE_MASK.get(h)
    if mask is None:
        print(f"  {h}: ⚠️ 마스크 없음")
        continue
    registered = [_LABELS[i] for i, v in enumerate(mask) if v > 0]
    print(f"  {h} ({int(mask.sum())}종): {registered}")

## 캐시 빌드 (CPU 런타임에서 실행)

> Run4/5에서 이미 빌드한 캐시를 재사용한다. 새로 빌드가 필요한 경우에만 실행.

In [ ]:
import gc

ws = DATASET_CFG["window"]["size"]
st = DATASET_CFG["window"]["stride"]
ec = DATASET_CFG["window"].get("event_context")
ss = DATASET_CFG["window"].get("steady_stride")

_ds_common = dict(
    gcs_fs=gcs,
    bucket_prefix=BUCKET_PREFIX, label_path=LABEL_PATH,
    window_size=ws, stride=st,
    event_context=ec, steady_stride=ss,
    cache_dir=CACHE_DIR, denoise=True,
    house_mask_dict=HOUSE_MASK,
)

for split_name, houses, weeks in [
    ("train", SPLIT["train"], [1, 2, 3, 4]),
    ("val",   SPLIT["val"],   [None]),
    ("test",  SPLIT["test"],  [None]),
]:
    for w in weeks:
        label = f"{split_name}(week={w})"
        print(f"\n[{label}] 빌드 중...")
        base = GCSNILMDataset(
            houses, week=w,
            fit_scaler=(split_name == "train" and w == 1),
            **_ds_common,
        )
        print(f"  base: {len(base):,} windows")
        tda = _NILMDatasetWithTDA(base, cache_dir=CACHE_DIR)
        print(f"  TDA:  완료")
        del tda, base
        gc.collect()

print("\n✅ 전체 캐시 빌드 완료")

## GPU 투입 전 캐시 확인

> GPU 런타임으로 전환 후 이 셀을 먼저 실행한다. 캐시 누락 시 CPU에서 캐시 빌드 셀을 재실행.

In [ ]:
import hashlib
from features.tda import TDA_DIM

ws = DATASET_CFG["window"]["size"]
st = DATASET_CFG["window"]["stride"]
ec = 0  # _NILMDatasetWithTDA 기본값
ss = 0  # _NILMDatasetWithTDA 기본값

def _gcs_week_key(week):
    return hashlib.md5(
        f"None|{week}|None|{BUCKET_PREFIX}|True".encode()
    ).hexdigest()[:8]

def _gcs_tda_key(houses, week):
    return hashlib.md5(
        f"{sorted(houses)}|None|{week}|None|{ws}|{st}|{BUCKET_PREFIX}|30|True".encode()
    ).hexdigest()[:12]

missing = []
for label, houses, week in [
    ("train_w1", SPLIT["train"], 1),
    ("train_w2", SPLIT["train"], 2),
    ("train_w3", SPLIT["train"], 3),
    ("train_w4", SPLIT["train"], 4),
    ("val",      SPLIT["val"],   None),
    ("test",     SPLIT["test"],  None),
]:
    wk    = _gcs_week_key(week)
    b_ok  = all((CACHE_DIR / f"nilm_gcs_{h}_{wk}.npz").exists() for h in houses)
    t_key = _gcs_tda_key(houses, week)
    t_ok  = (CACHE_DIR / f"tda_{t_key}_ec{ec}_ss{ss}_d{TDA_DIM}.pt").exists()
    flag  = "✅" if (b_ok and t_ok) else "❌"
    print(f"  {flag}  {label:10s}  base={'O' if b_ok else 'X'}  tda={'O' if t_ok else 'X'}")
    if not (b_ok and t_ok):
        missing.append(label)

if missing:
    raise RuntimeError(
        f"\n⛔ 캐시 누락: {missing}\n"
        "GPU 런타임을 끄고 CPU 런타임에서 캐시 빌드 셀을 먼저 실행하세요."
    )
else:
    print("\n✅ 모든 캐시 확인 완료 — GPU 학습 진행 가능")

## 함수 정의

> **Run 5 대비 변경점**: early stopping을 `val_mae` 단독 모니터링으로 수정.
> Run 5에서 `(f1_cls, -val_mae)` 복합 점수를 사용했을 때 logit gating으로 f1_cls가
> epoch 1에 포화되어 수렴 전 조기 종료된 문제를 수정.

In [ ]:
import json, time
import torch
import numpy as np
from torch.utils.data import DataLoader


def run_exp_gcs(exp_name: str, model_name: str,
                denoise: bool = True, tag: str = "", val_week=None) -> dict:
    """GCS 데이터 + house_mask 조건화 학습. early stopping = val_mae 단독(↓)."""
    exp_cfg = TRAIN_CFG["experiments"][exp_name]
    week = exp_cfg.get("week")
    ws   = DATASET_CFG["window"]["size"]
    st   = DATASET_CFG["window"]["stride"]
    ec   = DATASET_CFG["window"].get("event_context")
    ss   = DATASET_CFG["window"].get("steady_stride")
    bs   = TRAIN_CFG["training"]["batch_size"]
    ep   = TRAIN_CFG["training"]["epochs"]
    pat  = TRAIN_CFG["training"]["early_stopping_patience"]
    lr   = TRAIN_CFG["training"]["learning_rate"]
    wd   = TRAIN_CFG["optimizer"]["weight_decay"]
    lambda_mse = TRAIN_CFG["training"]["loss_weights"]["mse"]
    _suffix = f"_{tag}" if tag else ""
    _label  = f"{exp_name}/{model_name}" + (f"[{tag}]" if tag else "")

    print(f"\n{'='*58}")
    print(f"  {_label}  week={week}  denoise={denoise}")
    print(f"{'='*58}")

    print("  [1/4] 데이터셋 로딩...")

    resume_exp = exp_cfg.get("resume_from")
    prev_scaler = None
    if resume_exp:
        scaler_path = CKPT_DIR / f"{resume_exp}_{model_name}{_suffix}_scaler.json"
        if scaler_path.exists():
            prev_scaler = PowerScaler.load(scaler_path)
            print(f"  └─ scaler 로드: mean={prev_scaler.mean:.2f}W  std={prev_scaler.std:.2f}W")

    _ds_common = dict(
        gcs_fs=gcs,
        bucket_prefix=BUCKET_PREFIX, label_path=LABEL_PATH,
        window_size=ws, stride=st,
        event_context=ec, steady_stride=ss,
        cache_dir=CACHE_DIR, denoise=denoise,
        house_mask_dict=HOUSE_MASK,
    )

    if prev_scaler is not None:
        base_train = GCSNILMDataset(SPLIT["train"], scaler=prev_scaler, week=week, **_ds_common)
    else:
        base_train = GCSNILMDataset(SPLIT["train"], fit_scaler=True, week=week, **_ds_common)
    train_ds = _NILMDatasetWithTDA(base_train, cache_dir=CACHE_DIR)
    print(f"  train(week{week}): {len(train_ds):,} windows")

    base_val = GCSNILMDataset(SPLIT["val"], scaler=base_train.scaler, week=val_week, **_ds_common)
    val_ds   = _NILMDatasetWithTDA(base_val, cache_dir=CACHE_DIR)

    train_dl = DataLoader(train_ds, batch_size=bs, shuffle=True,  num_workers=2, pin_memory=False)
    val_dl   = DataLoader(val_ds,   batch_size=bs, shuffle=False, num_workers=2, pin_memory=False)
    print(f"  val:   {len(val_ds):,} windows")

    from classifier.label_map import get_on_thresholds as _get_on_thr
    _raw_thr   = np.array(_get_on_thr(), dtype=np.float32)
    _scl       = base_train.scaler
    _fixed_thr = (_raw_thr - _scl.mean) / _scl.std if _scl is not None else None

    print("  [2/4] 모델 초기화...")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"  device: {device}")
    model = build_model(model_name, ws).to(device)

    if resume_exp:
        prev_ckpt = CKPT_DIR / f"{resume_exp}_{model_name}{_suffix}.pt"
        if prev_ckpt.exists():
            _ckpt  = torch.load(prev_ckpt, map_location=device, weights_only=True)
            _state = _ckpt["model_state"] if isinstance(_ckpt, dict) and "model_state" in _ckpt else _ckpt
            model.load_state_dict(_state)
            print(f"  └─ 모델 로드: {prev_ckpt.name}")
        else:
            print(f"  └─ 경고: {prev_ckpt.name} 없음 — 처음부터 학습")

    optimizer  = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    scheduler  = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, "min",
        factor=TRAIN_CFG["scheduler"]["factor"],
        patience=TRAIN_CFG["scheduler"]["patience"],
    )
    amp_scaler = torch.amp.GradScaler("cuda") if device.type == "cuda" else None

    _app_index = {name: i for i, name in enumerate(APPLIANCE_LABELS)}
    _scale_cfg = TRAIN_CFG.get("appliance_loss_scale", {})
    appliance_scale = torch.ones(len(APPLIANCE_LABELS), device=device)
    for _name, _s in _scale_cfg.items():
        if _name in _app_index:
            appliance_scale[_app_index[_name]] = float(_s)
            print(f"  appliance_loss_scale [{_name}]: ×{_s}")

    pos_weight_max = float(TRAIN_CFG["training"].get("pos_weight_max", 20.0))
    print("  pos_weight 계산 중...")
    pos_weight = compute_pos_weight(train_dl, device, max_weight=pos_weight_max)
    for name, pw in zip(APPLIANCE_LABELS, pos_weight.cpu().tolist()):
        print(f"    {name}: {pw:.2f}")

    print("  [3/4] 학습 시작...")
    # ── early stopping: val_mae 단독 (↓), Run5의 (f1_cls, -val_mae) 복합 점수 버그 수정 ──
    best_score          = float("inf")
    best_cls_thresholds = None
    best_vm             = None
    best_state          = None
    no_improve          = 0
    t0 = time.perf_counter()

    for epoch in range(1, ep + 1):
        loss = train_one_epoch(model, train_dl, optimizer, model_name, device,
                               amp_scaler=amp_scaler, pos_weight=pos_weight,
                               lambda_mse=lambda_mse, appliance_scale=appliance_scale)
        vm   = evaluate(model, val_dl, model_name, device, cls_thresholds=_fixed_thr)
        scheduler.step(vm["mae"])
        lr_now = optimizer.param_groups[0]["lr"]
        f1_cls_str = f"  f1_cls={vm['f1_cls']:.3f}" if vm.get("f1_cls") is not None else ""
        print(f"  ep {epoch:3d}/{ep}  loss={loss:.4f}  "
              f"val_mae={vm['mae']:.2f}W  f1={vm['f1']:.3f}{f1_cls_str}  lr={lr_now:.2e}")

        _score = vm["mae"]
        if _score < best_score or best_state is None:
            best_score          = _score
            best_cls_thresholds = vm.get("best_cls_thresholds")
            best_vm             = vm
            best_state          = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_improve          = 0
        else:
            no_improve += 1
            if no_improve >= pat:
                print(f"  조기 종료 ({pat} epoch 개선 없음)")
                break

    elapsed = time.perf_counter() - t0

    print("  [4/4] 체크포인트 & 메트릭 저장...")
    if best_state:
        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})

    torch.save({"model_state": model.state_dict(), "best_cls_thresholds": best_cls_thresholds},
               CKPT_DIR / f"{exp_name}_{model_name}{_suffix}.pt")
    if base_train.scaler is not None:
        base_train.scaler.save(CKPT_DIR / f"{exp_name}_{model_name}{_suffix}_scaler.json")

    fm = best_vm if best_vm is not None else vm
    fm.update({"exp": exp_name, "model": model_name, "week": week,
               "tag": tag, "denoise": denoise,
               "training_time_s": round(elapsed, 1),
               "final_lr": optimizer.param_groups[0]["lr"]})
    with open(RESULTS_DIR / f"{exp_name}_{model_name}{_suffix}_metrics.json", "w") as f:
        json.dump(fm, f, ensure_ascii=False, indent=2)

    f1_cls_str = f"  F1_cls={fm['f1_cls']:.3f}" if fm.get("f1_cls") is not None else ""
    print(f"  ✅  val MAE={fm['mae']:.2f}W  RMSE={fm['rmse']:.2f}W  "
          f"SAE={fm['sae']:.4f}  F1={fm['f1']:.3f}{f1_cls_str}  ({elapsed:.0f}s)")
    return fm


print("함수 정의 완료 — run_exp_gcs (val_mae 단독 early stopping)")

## Incremental 학습 (EXP1 → EXP2 → EXP3 → EXP4)

> 각 주차 학습 후 val MAE를 기록한다. 성능이 올라가기 시작하면 이전 체크포인트가 최적.
> 학습 완료 후 best week 선택 셀에서 자동으로 최적 주차를 선정한다.

In [ ]:
result_exp1 = run_exp_gcs("EXP1", "cnn_tda", denoise=True)

In [ ]:
result_exp2 = run_exp_gcs("EXP2", "cnn_tda", denoise=True)

In [ ]:
result_exp3 = run_exp_gcs("EXP3", "cnn_tda", denoise=True)

In [ ]:
result_exp4 = run_exp_gcs("EXP4", "cnn_tda", denoise=True)

## 최적 주차 선택

> 주차별 val MAE를 비교해 최저값 주차의 체크포인트를 `best_exp`로 선정한다.
> 이후 Test 평가는 `best_exp` 체크포인트로 수행한다.

In [ ]:
import json as _json

week_results = {
    "EXP1": result_exp1,
    "EXP2": result_exp2,
    "EXP3": result_exp3,
    "EXP4": result_exp4,
}

best_exp = min(week_results, key=lambda k: week_results[k]["mae"])
best_mae = week_results[best_exp]["mae"]

print("주차별 val MAE 추이:")
prev_mae = None
for exp, r in week_results.items():
    trend = ""
    if prev_mae is not None:
        diff = r["mae"] - prev_mae
        trend = f"  (↓{abs(diff):.2f}W 개선)" if diff < 0 else f"  (↑{abs(diff):.2f}W 악화)"
    marker = " ← best" if exp == best_exp else ""
    print(f"  {exp}: {r['mae']:.2f}W{trend}{marker}")
    prev_mae = r["mae"]

print(f"\n최적 체크포인트: {best_exp} (val MAE={best_mae:.2f}W)")

# 요약 저장
summary = {
    "best_exp": best_exp,
    "best_val_mae": best_mae,
    "per_week": {exp: {"val_mae": r["mae"], "val_f1": r["f1"]} for exp, r in week_results.items()},
}
with open(RESULTS_DIR / "run6_week_summary.json", "w") as _f:
    _json.dump(summary, _f, ensure_ascii=False, indent=2)
print("저장: run6_week_summary.json")

## Test 평가 (oracle threshold)

> `best_exp` 체크포인트로 test house_067 전 기간 평가.

In [ ]:
_model_name = "cnn_tda"
ckpt_path   = CKPT_DIR / f"{best_exp}_{_model_name}.pt"
scaler_path = CKPT_DIR / f"{best_exp}_{_model_name}_scaler.json"

if not ckpt_path.exists():
    print(f"체크포인트 없음: {ckpt_path}")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    ws = DATASET_CFG["window"]["size"]
    st = DATASET_CFG["window"]["stride"]
    ec = DATASET_CFG["window"].get("event_context")
    ss = DATASET_CFG["window"].get("steady_stride")
    bs = TRAIN_CFG["training"]["batch_size"]

    scaler = PowerScaler.load(scaler_path) if scaler_path.exists() else None
    base_test = GCSNILMDataset(
        SPLIT["test"], scaler=scaler, week=None,
        gcs_fs=gcs, bucket_prefix=BUCKET_PREFIX, label_path=LABEL_PATH,
        window_size=ws, stride=st, event_context=ec, steady_stride=ss,
        cache_dir=CACHE_DIR, denoise=True,
        house_mask_dict=HOUSE_MASK,
    )
    test_ds = _NILMDatasetWithTDA(base_test, cache_dir=CACHE_DIR)
    test_dl = DataLoader(test_ds, batch_size=bs, shuffle=False, num_workers=2, pin_memory=False)

    model = build_model(_model_name, ws).to(device)
    _ckpt  = torch.load(ckpt_path, map_location=device, weights_only=True)
    _state = _ckpt["model_state"] if isinstance(_ckpt, dict) and "model_state" in _ckpt else _ckpt
    _best_thr = _ckpt.get("best_cls_thresholds") if isinstance(_ckpt, dict) else None
    model.load_state_dict(_state)

    _oracle_thr = np.array(_best_thr, dtype=np.float32) if _best_thr is not None else None

    print("=" * 58)
    print(f"  Test 평가 — {best_exp} / oracle threshold")
    print("=" * 58)

    tm = evaluate(model, test_dl, _model_name, device, cls_thresholds=_oracle_thr)
    tm.update({"exp": best_exp, "model": _model_name, "week": "all", "split": "test"})

    out_path = RESULTS_DIR / f"{best_exp}_{_model_name}_test_oracle_metrics.json"
    with open(out_path, "w") as _f:
        import json as _json2
        _json2.dump(tm, _f, ensure_ascii=False, indent=2)

    print(f"  MAE={tm['mae']:.2f}W  RMSE={tm['rmse']:.2f}W  "
          f"SAE={tm['sae']:.4f}  F1={tm['f1']:.3f}  F1_cls={tm['f1_cls']:.3f}")
    print(f"  저장: {out_path.name}")

## 결과 요약 (Run3 / Run4 / Run5 / Run6 비교)

In [ ]:
import json as _json

RUN3_VAL_MAE  = 39.32;  RUN3_TEST_MAE  = 44.14
RUN4_VAL_MAE  = 44.09;  RUN4_TEST_MAE  = 46.12
RUN5_TEST_MAE = 47.09   # val은 early stopping 버그로 무효

print("=" * 58)
print("  Run 6 (incremental + house_mask, ES 수정) 결과")
print("=" * 58)

# 주차별 val MAE
print("\n주차별 val MAE:")
for exp, r in week_results.items():
    marker = " ← best" if exp == best_exp else ""
    print(f"  {exp}(week{r['week']}): {r['mae']:.2f}W{marker}")

# Best val 비교
print(f"\n  Best val MAE: {best_mae:.2f}W")
for label, ref in [("Run3", RUN3_VAL_MAE), ("Run4", RUN4_VAL_MAE)]:
    diff = best_mae - ref
    print(f"    vs {label}({ref:.2f}W): {'↓' if diff < 0 else '↑'} {abs(diff):.2f}W")

# Test 비교
test_p = RESULTS_DIR / f"{best_exp}_{_model_name}_test_oracle_metrics.json"
if test_p.exists():
    tm6 = _json.load(open(test_p))
    print(f"\n  Test MAE: {tm6['mae']:.2f}W  RMSE={tm6['rmse']:.2f}W  F1={tm6['f1']:.3f}")
    for label, ref in [("Run3", RUN3_TEST_MAE), ("Run4", RUN4_TEST_MAE), ("Run5", RUN5_TEST_MAE)]:
        diff = tm6['mae'] - ref
        print(f"    vs {label}({ref:.2f}W): {'↓' if diff < 0 else '↑'} {abs(diff):.2f}W")

print("\n판단 기준:")
print(f"  Run6 val MAE < Run3(39.32W): house_mask + incremental 상승 효과 확인")
print(f"  Run6 val MAE < Run4(44.09W): house_mask + ES 수정 효과 확인")
print(f"  Run6 val MAE ≈ Run3: 개선 없음 → CNN 아키텍처 확대(B안) 검토")